In [1]:
import os
import json
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

# ── Config — update paths to match your input datasets ───────────────────────
FEATURE_DIRS = {
    "train": [
        "/kaggle/input/datasets/kavishgrover/features-v1/features/train",
        "/kaggle/input/datasets/kavishgrover/features-v2/features/train",
        "/kaggle/input/datasets/kavishgrover/features-v3/features/train",
        "/kaggle/input/datasets/kavishgrover/features-v4/features/train",
        "/kaggle/input/datasets/kavishgrover/features-v5/features/train",
        "/kaggle/input/datasets/kavishgrover/features-v6/features/train",
    ],
    "val": [
        "/kaggle/input/datasets/kavishgrover/features-v6/features/val",
    ],
    "test": [
        "/kaggle/input/datasets/kavishgrover/features-v6/features/test",
        "/kaggle/input/datasets/kavishgrover/features-v7/features/test",
    ],
}

ACTIONS_JSON = {
    "train": "/kaggle/input/datasets/kavishgrover/bdd-oia/train_25k_images_actions.json",
    "val"  : "/kaggle/input/datasets/kavishgrover/bdd-oia/val_25k_images_actions.json",
    "test" : "/kaggle/input/datasets/kavishgrover/bdd-oia/test_25k_images_actions.json",
}
REASONS_JSON = {
    "train": "/kaggle/input/datasets/kavishgrover/bdd-oia/train_25k_images_reasons.json",
    "val"  : "/kaggle/input/datasets/kavishgrover/bdd-oia/val_25k_images_reasons.json",
    "test" : "/kaggle/input/datasets/kavishgrover/bdd-oia/test_25k_images_reasons.json",
}

TOP_K        = 10
NUM_ACTIONS  = 4
NUM_REASONS  = 21
LAMBDA       = 1.0
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 50
LR_STEP      = 10       # divide lr by 10 every 10 epochs
BATCH_SIZE   = 16
NUM_WORKERS  = 2
SAVE_PATH    = "/kaggle/working/model_best.pt"
device       = "cuda" if torch.cuda.is_available() else "cpu"

# ── Label loading helpers ─────────────────────────────────────────────────────
def load_label_lookups(split):
    """Returns two dicts: filename -> action tensor, filename -> reason tensor"""
    with open(ACTIONS_JSON[split]) as f:
        actions_data = json.load(f)
    with open(REASONS_JSON[split]) as f:
        reasons_data = json.load(f)

    id_to_filename = {img["id"]: img["file_name"] for img in actions_data["images"]}

    action_lookup = {}
    for ann in actions_data["annotations"]:
        cat = ann["category"]
        if len(cat) == 5 and cat[4] == 1:
            continue  # skip confuse
        label = cat[:4]  # drop confuse dim if present
        action_lookup[id_to_filename[ann["id"]]] = torch.tensor(label, dtype=torch.float32)

    reason_lookup = {}
    for entry in reasons_data:
        reason_lookup[entry["file_name"]] = torch.tensor(entry["reason"], dtype=torch.float32)

    return action_lookup, reason_lookup

# ── Dataset ───────────────────────────────────────────────────────────────────
class BDDOIADataset(Dataset):
    def __init__(self, split):
        self.action_lookup, self.reason_lookup = load_label_lookups(split)

        # Collect all .pt files across all batch directories for this split
        self.pt_files = []
        for d in FEATURE_DIRS[split]:
            if os.path.exists(d):
                self.pt_files += glob.glob(os.path.join(d, "*.pt"))

        # Filter to only files that have both action and reason labels
        self.pt_files = [
            f for f in self.pt_files
            if self._get_fname(f) in self.action_lookup
            and self._get_fname(f) in self.reason_lookup
        ]

        print(f"[{split}] {len(self.pt_files)} samples loaded")

    def _get_fname(self, pt_path):
        # .pt files are saved as filename.replace(".jpg", ".pt")
        return os.path.basename(pt_path).replace(".pt", ".jpg")

    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        data  = torch.load(self.pt_files[idx], weights_only=True)
        fname = self._get_fname(self.pt_files[idx])

        local_feats = data["local_feats"]   # (N, 2048, 7, 7)
        global_map  = data["global_map"]    # (1, 2048, 7, 7)
        action_label = self.action_lookup[fname]   # (4,)
        reason_label = self.reason_lookup[fname]   # (21,)

        return local_feats, global_map, action_label, reason_label

def collate_fn(batch):
    """Custom collate since local_feats has variable N per image"""
    local_feats_list  = [item[0] for item in batch]
    global_maps       = torch.stack([item[1] for item in batch])   # (B, 1, 2048, 7, 7)
    action_labels     = torch.stack([item[2] for item in batch])   # (B, 4)
    reason_labels     = torch.stack([item[3] for item in batch])   # (B, 21)
    return local_feats_list, global_maps, action_labels, reason_labels

# ── Model ─────────────────────────────────────────────────────────────────────
class GlobalModule(nn.Module):
    """Reduces (B, 1, 2048, 7, 7) → (B, 1, 256, 7, 7)"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        # x: (B, 1, 2048, 7, 7)
        B = x.shape[0]
        x = x.squeeze(1)          # (B, 2048, 7, 7)
        x = self.net(x)            # (B, 256, 7, 7)
        return x.unsqueeze(1)      # (B, 1, 256, 7, 7)


class AISM(nn.Module):
    """
    Action-Inducing Selection Module.
    Input : local (N, 2048, 7, 7) + global (1, 256, 7, 7) per image
    Output: top-k object-scene tensors (k, 2304, 7, 7)
    """
    def __init__(self, k=10):
        super().__init__()
        self.k = k
        # 3 conv layers: 2304 → 512 → 128 → 1, outputs score per object
        self.selector = nn.Sequential(
            nn.Conv2d(2304, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 1, kernel_size=3, padding=1),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))  # spatial pool for scoring

    def forward(self, local_feats, global_feat):
        N = local_feats.shape[0]
        k = self.k

        # Expand global to match N objects → (N, 256, 7, 7)
        global_expanded = global_feat.unsqueeze(0).expand(N, -1, -1, -1)

        # Concatenate along channel dim → (N, 2304, 7, 7)
        obj_scene = torch.cat([local_feats, global_expanded], dim=1)

        # Score each object
        scores = self.selector(obj_scene)          # (N, 1, 7, 7)
        scores = self.gap(scores).view(N)          # (N,)
        scores = torch.softmax(scores, dim=0)      # (N,)

        # Select top-k (or all if N < k)
        actual_k     = min(k, N)
        topk_indices = torch.topk(scores, actual_k).indices
        topk_tensors = obj_scene[topk_indices]     # (actual_k, 2304, 7, 7)

        # Zero-pad if N < k
        if actual_k < k:
            pad = torch.zeros(k - actual_k, topk_tensors.shape[1],
                              topk_tensors.shape[2], topk_tensors.shape[3],
                              device=topk_tensors.device)
            topk_tensors = torch.cat([topk_tensors, pad], dim=0)  # (k, 2304, 7, 7)

        return topk_tensors  # always (k, 2304, 7, 7)


class PredictionHead(nn.Module):
    """
    Global pool top-k tensors, flatten, 3 FC layers → action + reason
    Input : (k, 2304, 7, 7)
    Output: action logits (4,), reason logits (21,)
    """
    def __init__(self, k=10, in_channels=2304, num_actions=4, num_reasons=21):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        feat_dim = k * in_channels  # 10 * 2304 = 23040

        self.shared_fc = nn.Sequential(
            nn.Linear(feat_dim, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.action_head = nn.Linear(1024, num_actions)
        self.reason_head = nn.Linear(1024, num_reasons)

    def forward(self, topk_tensors):
        # topk_tensors: (k, 2304, 7, 7)
        pooled = self.gap(topk_tensors)        # (k, 2304, 1, 1)
        flat   = pooled.view(-1)               # (k*2304,) = (23040,)
        feat   = self.shared_fc(flat)          # (1024,)
        return self.action_head(feat), self.reason_head(feat)


class OIANet(nn.Module):
    def __init__(self, k=TOP_K):
        super().__init__()
        self.global_module   = GlobalModule()
        self.aism            = AISM(k=k)
        self.prediction_head = PredictionHead(k=k)

    def forward(self, local_feats_list, global_maps):
        """
        local_feats_list : list of B tensors, each (N_i, 2048, 7, 7)
        global_maps      : (B, 1, 2048, 7, 7)
        Returns          : action_logits (B, 4), reason_logits (B, 21)
        """
        # Global module processes full batch
        global_feats = self.global_module(global_maps)  # (B, 1, 256, 7, 7)

        action_logits_list = []
        reason_logits_list = []

        for i in range(len(local_feats_list)):
            local_i  = local_feats_list[i].to(device)     # (N_i, 2048, 7, 7)
            global_i = global_feats[i].squeeze(0)          # (256, 7, 7)

            topk     = self.aism(local_i, global_i)        # (k, 2304, 7, 7)
            a_logit, r_logit = self.prediction_head(topk)  # (4,), (21,)

            action_logits_list.append(a_logit)
            reason_logits_list.append(r_logit)

        action_logits = torch.stack(action_logits_list)    # (B, 4)
        reason_logits = torch.stack(reason_logits_list)    # (B, 21)
        return action_logits, reason_logits

# ── Training loop ─────────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train() if train else model.eval()
    total_loss  = 0.0
    action_loss_total = 0.0
    reason_loss_total = 0.0
    n_batches   = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for local_feats_list, global_maps, action_labels, reason_labels in tqdm(loader, leave=False):
            global_maps   = global_maps.to(device)
            action_labels = action_labels.to(device)
            reason_labels = reason_labels.to(device)

            action_logits, reason_logits = model(local_feats_list, global_maps)

            l_action = criterion(action_logits, action_labels)
            l_reason = criterion(reason_logits, reason_labels)
            loss     = l_action + LAMBDA * l_reason

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss        += loss.item()
            action_loss_total += l_action.item()
            reason_loss_total += l_reason.item()
            n_batches         += 1

    return (total_loss / n_batches,
            action_loss_total / n_batches,
            reason_loss_total / n_batches)

# ── Main ──────────────────────────────────────────────────────────────────────
print("Loading datasets...")
train_dataset = BDDOIADataset("train")
val_dataset   = BDDOIADataset("val")

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn)

model     = OIANet(k=TOP_K).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = StepLR(optimizer, step_size=LR_STEP, gamma=0.1)

print(f"\nDevice: {device}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Starting training for {EPOCHS} epochs...\n")

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_a, train_r = run_epoch(model, train_loader, criterion, optimizer, train=True)
    val_loss,   val_a,   val_r   = run_epoch(model, val_loader,   criterion, optimizer, train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} (A: {train_a:.4f}, R: {train_r:.4f}) | "
          f"Val Loss: {val_loss:.4f} (A: {val_a:.4f}, R: {val_r:.4f}) | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch"     : epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss"  : val_loss,
        }, SAVE_PATH)
        print(f"  ✓ Saved best model (val_loss={val_loss:.4f})")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {SAVE_PATH}")

Loading datasets...
[train] 16028 samples loaded
[val] 2259 samples loaded

Device: cuda
Train batches: 1002 | Val batches: 142
Starting training for 50 epochs...



Epoch 01/50 | Train Loss: 0.9388 (A: 0.6247, R: 0.3141) | Val Loss: 0.8491 (A: 0.5567, R: 0.2924) | LR: 0.001000
  ✓ Saved best model (val_loss=0.8491)


Epoch 02/50 | Train Loss: 0.8846 (A: 0.5784, R: 0.3062) | Val Loss: 0.8457 (A: 0.5485, R: 0.2972) | LR: 0.001000
  ✓ Saved best model (val_loss=0.8457)


Epoch 03/50 | Train Loss: 0.8592 (A: 0.5566, R: 0.3027) | Val Loss: 0.8474 (A: 0.5489, R: 0.2985) | LR: 0.001000


Epoch 04/50 | Train Loss: 0.8419 (A: 0.5430, R: 0.2989) | Val Loss: 0.8551 (A: 0.5600, R: 0.2951) | LR: 0.001000


...

Epoch 46/50 | Train Loss: 0.3507 (A: 0.1438, R: 0.2069) | Val Loss: 1.4190 (A: 1.1043, R: 0.3147) | LR: 0.000000


Epoch 47/50 | Train Loss: 0.3511 (A: 0.1441, R: 0.2070) | Val Loss: 1.4197 (A: 1.1049, R: 0.3149) | LR: 0.000000


Epoch 48/50 | Train Loss: 0.3514 (A: 0.1439, R: 0.2074) | Val Loss: 1.4217 (A: 1.1068, R: 0.3148) | LR: 0.000000


Epoch 49/50 | Train Loss: 0.3514 (A: 0.1442, R: 0.2072) | Val Loss: 1.4197 (A: 1.1050, R: 0.3147) | LR: 0.000000


Epoch 50/50 | Train Loss: 0.3508 (A: 0.1436, R: 0.2071) | Val Loss: 1.4220 (A: 1.1072, R: 0.3148) | LR: 0.000000

Training complete. Best val loss: 0.8282
Model saved to: /kaggle/working/model_best.pt
